# Grafici sulla demografia

Genera i grafici demografici compatibili con dati da API World Bank e segnala quelli che richiedono API UN WPP o ISTAT.

Fonte indicata nei grafici: API pubblica di riferimento - elaborazione Nazareno Lecis.

Le specifiche dei grafici sono definite in questo notebook; le funzioni generali di download, trasformazione e disegno stanno in `analisi/utils/grafici.py`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analisi.utils.grafici import (
    PAESI_MONDO,
    genera_grafici_supportati,
    specifica_in_attesa,
    specifica_rapporto,
    specifica_world_bank,
)


## Specifiche dei grafici

Questa lista appartiene alla categoria. I grafici con `specifica_in_attesa` restano in inventario finche non colleghiamo la relativa API pubblica.


In [ ]:
SPECIFICHE_GRAFICI = [
    specifica_world_bank("2.1", "65", "Popolazione totale - livello", "popolazione_totale_livello", "SP.POP.TOTL", "level"),
    specifica_world_bank("2.1", "66-67", "Popolazione totale - crescita media", "popolazione_totale_crescita_media", "SP.POP.TOTL", "growth_10y"),
    specifica_world_bank("2.1", "68", "Popolazione totale - crescita cumulata", "popolazione_totale_crescita_cumulata", "SP.POP.TOTL", "index"),
    specifica_rapporto("2.2", "70", "Popolazione 15-64 anni - livello", "popolazione_15_64_livello", "SP.POP.TOTL", "SP.POP.1564.TO.ZS", 0.01, operazione="product"),
    specifica_rapporto("2.2", "71-72", "Popolazione 15-64 anni - crescita media", "popolazione_15_64_crescita_media", "SP.POP.TOTL", "SP.POP.1564.TO.ZS", 0.01, tipo="growth_10y", operazione="product"),
    specifica_rapporto("2.2", "73", "Popolazione 15-64 anni - crescita cumulata", "popolazione_15_64_crescita_cumulata", "SP.POP.TOTL", "SP.POP.1564.TO.ZS", 0.01, tipo="index", operazione="product"),
    specifica_in_attesa("2.2", "74-77", "Popolazione 15-74 anni", "popolazione_15_74", "UN WPP API", "Richiede serie per fasce 15-74 via API demografica."),
    specifica_in_attesa("2.3", "79-82", "Popolazione prime age 25-49", "popolazione_prime_age_25_49", "UN WPP API", "Richiede serie per fasce 25-49 via API demografica."),
    specifica_rapporto("2.4", "84", "Bambini 0-14 anni - livello", "bambini_0_14_livello", "SP.POP.TOTL", "SP.POP.0014.TO.ZS", 0.01, operazione="product"),
    specifica_rapporto("2.4", "85-86", "Bambini 0-14 anni - crescita media", "bambini_0_14_crescita_media", "SP.POP.TOTL", "SP.POP.0014.TO.ZS", 0.01, tipo="growth_10y", operazione="product"),
    specifica_rapporto("2.4", "87-88", "Bambini 0-14 anni - crescita cumulata", "bambini_0_14_crescita_cumulata", "SP.POP.TOTL", "SP.POP.0014.TO.ZS", 0.01, tipo="index", operazione="product"),
    specifica_rapporto("2.5", "90", "Popolazione anziana 65+ - livello", "popolazione_anziana_livello", "SP.POP.TOTL", "SP.POP.65UP.TO.ZS", 0.01, operazione="product"),
    specifica_rapporto("2.5", "91-92", "Popolazione anziana 65+ - crescita media", "popolazione_anziana_crescita_media", "SP.POP.TOTL", "SP.POP.65UP.TO.ZS", 0.01, tipo="growth_10y", operazione="product"),
    specifica_rapporto("2.5", "93-94", "Popolazione anziana 65+ - crescita cumulata", "popolazione_anziana_crescita_cumulata", "SP.POP.TOTL", "SP.POP.65UP.TO.ZS", 0.01, tipo="index", operazione="product"),
    specifica_in_attesa("2.6", "96", "Popolazione femminile in eta fertile", "popolazione_femminile_eta_fertile", "UN WPP API", "Richiede serie femminili per eta fertile."),
    specifica_world_bank("2.7", "98-99", "Tasso di natalita", "tasso_natalita", "SP.DYN.CBRT.IN", "level"),
    specifica_world_bank("2.7", "100-101", "Fecondita - nati vivi per donna", "fecondita_nati_per_donna", "SP.DYN.TFRT.IN", "level"),
    specifica_world_bank("2.8", "103-105", "Rapporto dipendenza anziani", "rapporto_dipendenza_anziani", "SP.POP.DPND.OL", "level"),
    specifica_world_bank("2.8", "106-107", "Rapporto dipendenza bambini", "rapporto_dipendenza_bambini", "SP.POP.DPND.YG", "level"),
    specifica_world_bank("2.9", "110", "Popolazione 0-14 anni - % sul totale", "popolazione_0_14_quota", "SP.POP.0014.TO.ZS", "level"),
    specifica_world_bank("2.9", "114", "Popolazione 65+ anni - % sul totale", "popolazione_65_plus_quota", "SP.POP.65UP.TO.ZS", "level"),
    specifica_in_attesa("2.9", "109,111-113", "Struttura popolazione per eta dettagliata", "struttura_popolazione_eta", "UN WPP API", "Richiede fasce 0-4, 0-19, 25-44, 50-64."),
    specifica_world_bank("2.10", "116-118", "Saldo naturale proxy: natalita meno mortalita", "saldo_naturale_proxy", "SP.DYN.CBRT.IN", "level", note="Nel notebook il saldo naturale va letto insieme a SP.DYN.CDRT.IN."),
    specifica_world_bank("2.11", "121", "Saldo migratorio totale", "saldo_migratorio_totale", "SM.POP.NETM", "level"),
    specifica_in_attesa("2.11", "122-125", "Saldo migratorio italiani ed espatri", "saldo_migratorio_italiani", "ISTAT API", "Richiede dataset ISTAT migrazioni cittadini italiani."),
    specifica_world_bank("2.12", "127", "Aspettativa di vita", "aspettativa_vita", "SP.DYN.LE00.IN", "level"),
]

print(f"Specifiche nella categoria: {len(SPECIFICHE_GRAFICI)}")


## Generazione PNG

I file vengono salvati nella cartella della categoria. Le tabelle restano in memoria.


In [ ]:
percorsi, inventario = genera_grafici_supportati(SPECIFICHE_GRAFICI, Path(ROOT / "analisi/demografia"), start_year=1960)
print(f"PNG generati: {len(percorsi)}")
for percorso in percorsi:
    print(percorso)
inventario


## Plot dei grafici generati

I grafici prodotti nella cella precedente vengono visualizzati qui sotto.


In [ ]:
from IPython.display import Image, display

for percorso in percorsi:
    display(Image(filename=str(percorso)))
